In [ ]:
import glob
import os
from pathlib import Path
import re
import pandas as pd
import numpy as np
import xarray as xr

def extract_dt(path):
    fname = Path(path).name
    match = re.search(r"DMV_GOES_image_(\d{12})\.nc$", fname)
    if not match:
        raise ValueError(f"Could not parse datetime from filename: {fname}")
    return pd.to_datetime(match.group(1), format="%Y%m%d%H%M")



# Sort files by timestamp
files_sorted = sorted(files, key=extract_dt)
times = pd.DatetimeIndex([extract_dt(f) for f in files_sorted])

# Expected range
start_expected = pd.Timestamp("2023-05-23 00:00")
end_expected = pd.Timestamp("2023-10-07 23:50")  # through Oct 8 inclusive
expected = pd.date_range(start_expected, end_expected, freq="10min")

print("First file:", files_sorted[0])
print("First timestamp:", times[0])
print("Last file:", files_sorted[-1])
print("Last timestamp:", times[-1])
print("Number of files:", len(times))
print("Expected number:", len(expected))

# Checks
print("Starts correctly:", times[0] == start_expected)
print("Ends correctly:", times[-1] == end_expected)
print(
  "10-min cadence:",
  (times.to_series().diff().dropna() == pd.Timedelta(minutes=10)).all()
  )
print("Complete sequence:", times.equals(expected))

# Diagnose missing or extra times
missing = expected.difference(times)
extra = times.difference(expected)

print("Missing timestamps:", len(missing))
print("Extra timestamps:", len(extra))

if len(missing) > 0:
    print("First few missing:")
    print(missing[:20])

if len(extra) > 0:
    print("First few extra:")
    print(extra[:20])

ds = xr.open_mfdataset(
    files,
    combine="nested",
    concat_dim="time",
)

"""

In [ ]:


# ---------------------------------------------------------------------
# Raw dataset timestamps
# ---------------------------------------------------------------------

times_raw = pd.DatetimeIndex(pd.to_datetime(ds["datetime"].values, utc=True))


times_nearest_10min = times_raw.round("10min")

# Offset between raw timestamp and nearest 10-minute timestamp.
offset_from_nearest = times_raw - times_nearest_10min

# ---------------------------------------------------------------------
# Expected timestamp range
# ---------------------------------------------------------------------
start_expected = pd.Timestamp("2023-05-23 00:00", tz="UTC")
end_expected = pd.Timestamp("2023-10-07 23:50", tz="UTC")

expected = pd.date_range(
    start=start_expected,
    end=end_expected,
    freq="10min",
)

tolerance = pd.Timedelta(seconds=60)

# ---------------------------------------------------------------------
# Per-sample diagnostics
# ---------------------------------------------------------------------
sample_df = pd.DataFrame({
    "time_index": np.arange(len(times_raw)),
    "raw_datetime": times_raw,
    "nearest_10min": times_nearest_10min,
    "offset_from_nearest": offset_from_nearest,
    "offset_seconds": offset_from_nearest.total_seconds(),
    "abs_offset_seconds": np.abs(offset_from_nearest.total_seconds()),
})

sample_df["nearest_10min_in_expected_range"] = sample_df["nearest_10min"].isin(expected)

sample_df["within_tolerance"] = (
    sample_df["abs_offset_seconds"] <= tolerance.total_seconds()
)

sample_df["valid_expected_sample"] = (
    sample_df["nearest_10min_in_expected_range"]
    & sample_df["within_tolerance"]
)

# ---------------------------------------------------------------------
# Identify extra samples before cleaning
# ---------------------------------------------------------------------

extra_offgrid_samples_df = sample_df[
    ~sample_df["valid_expected_sample"]
].copy()

# Valid samples:
valid_sample_df = sample_df[
    sample_df["valid_expected_sample"]
].copy()

# Duplicates within valid expected bins:
# First valid sample in each 10-min bin is kept; later samples are extras.
valid_sample_df["occurrence_in_bin"] = (
    valid_sample_df.groupby("nearest_10min").cumcount()
)

duplicate_valid_samples_df = valid_sample_df[
    valid_sample_df["occurrence_in_bin"] > 0
].copy()

# Combined extras:
extra_samples_df = pd.concat(
    [
        extra_offgrid_samples_df.assign(extra_reason="off_grid_or_out_of_range"),
        duplicate_valid_samples_df.assign(extra_reason="duplicate_valid_bin"),
    ],
    ignore_index=True,
).sort_values("time_index")

# ---------------------------------------------------------------------
# Missing expected samples before reindexing
# ---------------------------------------------------------------------
valid_bins = pd.DatetimeIndex(
    valid_sample_df["nearest_10min"].to_numpy()
)

missing = expected.difference(valid_bins)

expected_index_lookup = pd.Series(
    np.arange(len(expected)),
    index=expected,
)

missing_df = pd.DataFrame({
    "missing_datetime": missing,
    "expected_index": expected_index_lookup.loc[missing].values,
})

# Add previous/next actual dataset sample around each missing expected timestamp.
prev_indices = []
next_indices = []
prev_raw_datetimes = []
next_raw_datetimes = []
prev_nearest_10min = []
next_nearest_10min = []

for t in missing:
    insert_pos = times_nearest_10min.searchsorted(t)

    if insert_pos > 0:
        prev_indices.append(insert_pos - 1)
        prev_raw_datetimes.append(times_raw[insert_pos - 1])
        prev_nearest_10min.append(times_nearest_10min[insert_pos - 1])
    else:
        prev_indices.append(np.nan)
        prev_raw_datetimes.append(pd.NaT)
        prev_nearest_10min.append(pd.NaT)

    if insert_pos < len(times_nearest_10min):
        next_indices.append(insert_pos)
        next_raw_datetimes.append(times_raw[insert_pos])
        next_nearest_10min.append(times_nearest_10min[insert_pos])
    else:
        next_indices.append(np.nan)
        next_raw_datetimes.append(pd.NaT)
        next_nearest_10min.append(pd.NaT)

missing_df["previous_time_index"] = prev_indices
missing_df["previous_raw_datetime"] = prev_raw_datetimes
missing_df["previous_nearest_10min"] = prev_nearest_10min
missing_df["next_time_index"] = next_indices
missing_df["next_raw_datetime"] = next_raw_datetimes
missing_df["next_nearest_10min"] = next_nearest_10min

# ---------------------------------------------------------------------
# Cadence diagnostics before cleaning
# ---------------------------------------------------------------------
cadence_df = pd.DataFrame({
    "time_index": np.arange(len(times_raw)),
    "raw_datetime": times_raw,
    "nearest_10min": times_nearest_10min,
    "raw_delta_from_previous": times_raw.to_series().diff().values,
    "nearest_10min_delta_from_previous": pd.Series(times_nearest_10min).diff().values,
})

raw_cadence_breaks_df = cadence_df[
    cadence_df["raw_delta_from_previous"].notna()
    & (cadence_df["raw_delta_from_previous"] != pd.Timedelta(minutes=10))
].copy()

nearest_10min_cadence_breaks_df = cadence_df[
    cadence_df["nearest_10min_delta_from_previous"].notna()
    & (cadence_df["nearest_10min_delta_from_previous"] != pd.Timedelta(minutes=10))
].copy()

# ---------------------------------------------------------------------
# Keep only the first valid sample in each expected 10-minute bin
# ---------------------------------------------------------------------
valid_first_sample_df = valid_sample_df[
    valid_sample_df["occurrence_in_bin"] == 0
].copy()

keep_time_indices = valid_first_sample_df["time_index"].to_numpy()

# Subset original dataset to valid samples only.

ds_clean = ds.isel(time=keep_time_indices)

# Assign clean 10-minute timestamps to retained samples.

clean_time_index = pd.DatetimeIndex(
    valid_first_sample_df["nearest_10min"].to_numpy()
)

ds_clean = ds_clean.assign_coords(
    time=clean_time_index,
    raw_datetime=("time", valid_first_sample_df["raw_datetime"].to_numpy()),
    offset_seconds=("time", valid_first_sample_df["offset_seconds"].to_numpy()),
)

# Sort by clean time just in case.
ds_clean = ds_clean.sortby("time")

# ---------------------------------------------------------------------
# Reindex to complete expected sequence
# ---------------------------------------------------------------------

ds_complete = ds_clean.reindex(time=expected)

# ---------------------------------------------------------------------
# Add useful coordinates and sample-presence flag
# ---------------------------------------------------------------------

present_mask = ds_complete["raw_datetime"].notnull()

ds_complete = ds_complete.assign_coords(
    datetime_10min=("time", expected),
    sample_present=("time", present_mask.data),
)


ds_complete = ds_complete.assign_coords(
    datetime=("time", expected.astype(str))
)

# Missing samples after reindexing
missing_after_reindex = expected[~present_mask.values]

missing_after_reindex_df = pd.DataFrame({
    "missing_datetime": missing_after_reindex,
    "expected_index": np.where(~present_mask.values)[0],
})

# ---------------------------------------------------------------------
# Print summary
# ---------------------------------------------------------------------
print("Original dataset samples:", ds.sizes["time"])
print("Expected samples:", len(expected))
print("Valid retained samples:", ds_clean.sizes["time"])
print("Complete dataset samples:", ds_complete.sizes["time"])

print("Removed extra/off-grid/out-of-range samples:", len(extra_offgrid_samples_df))
print("Removed duplicate valid-bin samples:", len(duplicate_valid_samples_df))
print("Total removed samples:", len(extra_samples_df))

print("Inserted NaN missing samples:", len(missing_after_reindex_df))

print("\nFirst few removed samples:")
print(extra_samples_df.head(20))

print("\nFirst few inserted missing samples:")
print(missing_after_reindex_df.head(20))

print("\nFirst few raw cadence breaks:")
print(raw_cadence_breaks_df.head(20))

print("\nFirst few nearest-10min cadence breaks:")
print(nearest_10min_cadence_breaks_df.head(20))

# ---------------------------------------------------------------------
# Final checks before saving
# ---------------------------------------------------------------------
final_times = pd.DatetimeIndex(ds_complete["time"].values)

print("\nFinal checks before serialization patch")
print("Starts correctly:", final_times[0] == start_expected)
print("Ends correctly:", final_times[-1] == end_expected)
print("Expected length:", len(final_times) == len(expected))
print("Complete sequence:", final_times.equals(expected))

# =====================================================================
# SERIALIZATION PATCH RIGHT BEFORE TO_NETCDF
# =====================================================================



def to_naive_utc_datetime64(values):
    '''
    Convert datetime-like values to timezone-naive UTC datetime64[ns].

    Handles:
    - timezone-aware pandas timestamps
    - timezone-naive datetime64
    - NaT values from reindexing
    '''
    dt = pd.to_datetime(values, utc=True, errors="coerce")
    return pd.DatetimeIndex(dt).tz_localize(None).to_numpy(dtype="datetime64[ns]")


# Save a copy so diagnostics in memory remain available if needed.
ds_to_save = ds_complete.copy()

# Convert main time coordinate.
ds_to_save = ds_to_save.assign_coords(
    time=to_naive_utc_datetime64(ds_to_save["time"].values)
)

# Convert datetime_10min coordinate.
if "datetime_10min" in ds_to_save.coords:
    ds_to_save = ds_to_save.assign_coords(
        datetime_10min=("time", to_naive_utc_datetime64(ds_to_save["datetime_10min"].values))
    )

# Convert raw_datetime coordinate.

if "raw_datetime" in ds_to_save.coords:
    ds_to_save = ds_to_save.assign_coords(
        raw_datetime=("time", to_naive_utc_datetime64(ds_to_save["raw_datetime"].values))
    )

# Convert string datetime coordinate to plain fixed-width unicode strings.
# This avoids object dtype serialization issues.
if "datetime" in ds_to_save.coords:
    datetime_str = pd.DatetimeIndex(ds_to_save["time"].values).astype(str).to_numpy()
    ds_to_save = ds_to_save.assign_coords(
        datetime=("time", datetime_str.astype("U19"))
    )

# spatial_ref warning fix:

if "spatial_ref" in ds_to_save:
    if "time" in ds_to_save["spatial_ref"].dims:
        ds_to_save["spatial_ref"] = ds_to_save["spatial_ref"].astype("float64")


for var_name in list(ds_to_save.data_vars):
    var = ds_to_save[var_name]
    if "time" in var.dims and np.issubdtype(var.dtype, np.integer):
        if var.isnull().any():
            ds_to_save[var_name] = var.astype("float64")


for name in ds_to_save.variables:
    ds_to_save[name].encoding = {}


encoding = {}
for var_name in ds_to_save.data_vars:
    encoding[var_name] = {
        "zlib": True,
        "complevel": 4,
    }

# ---------------------------------------------------------------------
# Save cleaned complete dataset
# ---------------------------------------------------------------------

ds_to_save.to_netcdf(
    out_path,
    encoding=encoding,
)

print(f"\nSaved cleaned complete dataset to: {out_path}")

# ---------------------------------------------------------------------
# Quick reopen check
# ---------------------------------------------------------------------
ds_check = xr.open_dataset(out_path)

print("\nSaved file check")
print(ds_check)
print("Saved samples:", ds_check.sizes["time"])
print("Saved first time:", pd.to_datetime(ds_check["time"].values[0]))
print("Saved last time:", pd.to_datetime(ds_check["time"].values[-1]))


In [ ]:
def preprocess_goes_dataset(
    ds,
    channel_vars=None,
    fill_limit_steps=12,
    spatial_fill_cells=2,
    mask_var="sample_present",
    time_dim="time",
    y_dim="y",
    x_dim="x",
    raw_min=150.0,
    raw_max=350.0,
    slow_time_smooth_days=21,
    baseline_spatial_smooth_cells=1,
    diurnal_time_smooth_steps=2,
    anom_clip_method="mad",
    anom_mad_threshold=8.0,
    anom_abs_limit=80.0,
    return_clipped_anom=True,
):
    """
    Preprocess GOES image variables from an xarray.Dataset.

    Expected input:
        ds[GOES_channel](time, y, x)

    Important:
        This function does NOT smooth goes_filled.
        It only smooths the estimated baseline components:
            slow_component
            diurnal_cycle

    Output variables:
        goes_original
        goes_filled
        slow_component
        diurnal_cycle
        anom_raw
        anom
        anom_valid_mask

    Output dimensions:
        time, channel, y, x
    """

    if channel_vars is None:
        channel_vars = [
            "GOES_C13_LWIR",
            "GOES_C14_LWIR",
            "GOES_C15_LWIR",
            "GOES_C16_LWIR",
        ]

    missing = [v for v in channel_vars if v not in ds.data_vars]
    if missing:
        raise ValueError(f"These channel variables are missing from ds: {missing}")

    ds = ds.copy()
    ds[time_dim] = pd.to_datetime(ds[time_dim].values)

    # Apply sample_present mask before stacking channels
    if mask_var is not None and mask_var in ds:
        valid_mask = ds[mask_var].astype(bool)
        ds_goes = ds[channel_vars].where(valid_mask)
    else:
        ds_goes = ds[channel_vars]

    # Stack separate channel variables into one channel dimension
    da = xr.concat(
        [ds_goes[v].astype("float32") for v in channel_vars],
        dim=xr.IndexVariable("channel", channel_vars),
    )

    da = da.transpose(time_dim, "channel", y_dim, x_dim)
    da.name = "goes_original"

    # Mask physically suspicious raw brightness temperatures
    if raw_min is not None:
        da = da.where(da >= raw_min)

    if raw_max is not None:
        da = da.where(da <= raw_max)

    # Fill small spatial holes only
    goes_spatial_filled = (
        da
        .interpolate_na(
            dim=x_dim,
            method="linear",
            limit=spatial_fill_cells,
            use_coordinate=False,
        )
        .interpolate_na(
            dim=y_dim,
            method="linear",
            limit=spatial_fill_cells,
            use_coordinate=False,
        )
    )

    # Fill short temporal gaps only
    # With 10-minute data and fill_limit_steps=12, this fills up to ~2 hours.
    goes_filled = goes_spatial_filled.interpolate_na(
        dim=time_dim,
        method="linear",
        limit=fill_limit_steps,
        use_coordinate=True,
    )
    goes_filled.name = "goes_filled"

    # Estimate slow background from daily means
    daily_mean = goes_filled.resample({time_dim: "1D"}).mean()

    daily_smooth = (
        daily_mean
        .rolling(
            {time_dim: slow_time_smooth_days},
            center=True,
            min_periods=max(5, slow_time_smooth_days // 3),
        )
        .mean()
    )

    slow_component = daily_smooth.interp({time_dim: goes_filled[time_dim]})

    # Smooth only the slow baseline spatially, not goes_filled
    if baseline_spatial_smooth_cells is not None and baseline_spatial_smooth_cells > 0:
        spatial_window = 2 * baseline_spatial_smooth_cells + 1

        slow_component = (
            slow_component
            .rolling(
                {y_dim: spatial_window, x_dim: spatial_window},
                center=True,
                min_periods=1,
            )
            .mean()
        )

    slow_component = slow_component.transpose(time_dim, "channel", y_dim, x_dim)
    slow_component.name = "slow_component"

    # Remove slow background
    detrended = goes_filled - slow_component

    # Estimate mean diurnal cycle by clock time
    diurnal_cycle_by_time = detrended.groupby(f"{time_dim}.time").mean(time_dim)

    if time_dim in diurnal_cycle_by_time.dims:
        diurnal_cycle_by_time = diurnal_cycle_by_time.rename({time_dim: "time_of_day"})

    # Smooth only the diurnal baseline across neighboring clock-time bins
    if diurnal_time_smooth_steps is not None and diurnal_time_smooth_steps > 0:
        diurnal_window = 2 * diurnal_time_smooth_steps + 1

        diurnal_cycle_by_time = (
            diurnal_cycle_by_time
            .rolling(
                {"time_of_day": diurnal_window},
                center=True,
                min_periods=1,
            )
            .mean()
        )

    time_of_day_indexer = xr.DataArray(
        goes_filled[time_dim].dt.time.values,
        dims=time_dim,
        coords={time_dim: goes_filled[time_dim]},
        name="time_of_day",
    )

    diurnal_cycle = diurnal_cycle_by_time.sel(
        time_of_day=time_of_day_indexer
    )

    diurnal_cycle = diurnal_cycle.transpose(time_dim, "channel", y_dim, x_dim)

    # Smooth only the diurnal baseline spatially, not goes_filled
    if baseline_spatial_smooth_cells is not None and baseline_spatial_smooth_cells > 0:
        spatial_window = 2 * baseline_spatial_smooth_cells + 1

        diurnal_cycle = (
            diurnal_cycle
            .rolling(
                {y_dim: spatial_window, x_dim: spatial_window},
                center=True,
                min_periods=1,
            )
            .mean()
        )

    diurnal_cycle = diurnal_cycle.transpose(time_dim, "channel", y_dim, x_dim)
    diurnal_cycle.name = "diurnal_cycle"

    # Raw anomaly before outlier masking
    anom_raw = detrended - diurnal_cycle
    anom_raw.name = "anom_raw"

    # Robust outlier mask for anomalies
    if anom_clip_method == "mad":
        median = anom_raw.median(dim=(time_dim, y_dim, x_dim), skipna=True)

        mad = np.abs(anom_raw - median).median(
            dim=(time_dim, y_dim, x_dim),
            skipna=True,
        )

        robust_sigma = 1.4826 * mad
        robust_sigma = robust_sigma.where(robust_sigma > 0)

        robust_z = np.abs((anom_raw - median) / robust_sigma)
        anom_valid_mask = robust_z <= anom_mad_threshold

    elif anom_clip_method == "quantile":
        q_low = anom_raw.quantile(
            0.001,
            dim=(time_dim, y_dim, x_dim),
            skipna=True,
        )

        q_high = anom_raw.quantile(
            0.999,
            dim=(time_dim, y_dim, x_dim),
            skipna=True,
        )

        if "quantile" in q_low.coords:
            q_low = q_low.drop_vars("quantile")

        if "quantile" in q_high.coords:
            q_high = q_high.drop_vars("quantile")

        anom_valid_mask = (anom_raw >= q_low) & (anom_raw <= q_high)

    elif anom_clip_method is None:
        anom_valid_mask = xr.ones_like(anom_raw, dtype=bool)

    else:
        raise ValueError("anom_clip_method must be one of: 'mad', 'quantile', None")

    if anom_abs_limit is not None:
        anom_valid_mask = anom_valid_mask & (np.abs(anom_raw) <= anom_abs_limit)

    anom_valid_mask.name = "anom_valid_mask"

    if return_clipped_anom:
        anom = anom_raw.where(anom_valid_mask)
    else:
        anom = anom_raw

    anom.name = "anom"

    out = xr.Dataset(
        {
            "goes_original": da,
            "goes_filled": goes_filled,
            "slow_component": slow_component,
            "diurnal_cycle": diurnal_cycle,
            "anom_raw": anom_raw,
            "anom": anom,
            "anom_valid_mask": anom_valid_mask,
        }
    )

    if "time_of_day" in out.coords or "time_of_day" in out.variables:
        out = out.drop_vars("time_of_day")

    # Add circular time-of-day predictors from the real datetime coordinate.
    tod_minutes = (
        out[time_dim].dt.hour * 60
        + out[time_dim].dt.minute
        + out[time_dim].dt.second / 60
    ).astype("float32")

    tod_rad = 2.0 * np.pi * tod_minutes / 1440.0

    out["hour_sin"] = np.sin(tod_rad).astype("float32")
    out["hour_cos"] = np.cos(tod_rad).astype("float32")

    out["hour_sin"].attrs = {
        "long_name": "sine of time of day",
        "description": "sin(2*pi*minutes_since_midnight/1440)",
    }

    out["hour_cos"].attrs = {
        "long_name": "cosine of time of day",
        "description": "cos(2*pi*minutes_since_midnight/1440)",
    }

    return out

In [ ]:
out_path = "/glade/derecho/scratch/goes_files/"
ds = xr.open_dataset(
    out_path+"goes_concat_2023_clean_complete.nc"
)

In [ ]:
ds_pp = preprocess_goes_dataset(
    ds,
    slow_time_smooth_days=21,
    baseline_spatial_smooth_cells=1,
    diurnal_time_smooth_steps=2,
    anom_abs_limit=80.0,
)